# NLI por Passos — Ground Truth × Gerações Gherkin (JSON)

Este notebook calcula **somente NLI por passos Gherkin**. Não é realizado NLI global do cenário.

O objetivo é produzir, para cada cenário gerado em cada execução, um único `nli_score` que represente sua **proximidade semântica com o cenário de referência**, preservando também indicadores de completude e contradição.

## Estratégia

1. Extrair `Given`, `When`, `Then`, `And` e `But` do ground truth e da geração.
2. `And`/`But` herdam o contexto semântico anterior (`Given`, `When` ou `Then`).
3. Alinhar os passos por **grupo semântico + posição dentro do grupo**.
4. Para cada par alinhado, calcular NLI nas duas direções: referência → geração e geração → referência.
5. Para cada direção:

\[
S(A,B)=P(E)+0.5P(N)
\]

Como \(P(E)+P(N)+P(C)=1\), também:

\[
S(A,B)=\frac{1+P(E)-P(C)}{2}
\]

6. O score bidirecional do passo \(i\) é:

\[
NLI_i=\frac{S(R_i,G_i)+S(G_i,R_i)}{2}
\]

7. O score final do cenário é:

\[
NLI_{cenario}=\frac{1}{m}\sum_{i=1}^{m}NLI_i
\]

onde \(m\) é a quantidade de passos alinhados.

**Quanto maior o `nli_score`, maior a proximidade semântica.**

## Cobertura

A cobertura é reportada separadamente:

\[
coverage=\frac{passos\ alinhados}{passos\ do\ ground\ truth}
\]

Passos extras não aumentam a cobertura e são contabilizados separadamente.

## Saída

O notebook gera **um único CSV**, com uma linha por cenário/execução. Não é calculado nem exportado NLI global.

In [ ]:
# ============================================================
# 1. INSTALAÇÃO DAS DEPENDÊNCIAS
# ============================================================

!pip -q install -U transformers sentencepiece accelerate safetensors

In [ ]:
# ============================================================
# 2. IMPORTAÇÕES E CONFIGURAÇÃO
# ============================================================

import json
import re
import os
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForSequenceClassification

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)

MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# NLI é aplicado a passos, não ao cenário inteiro.
MAX_LENGTH = 192
NLI_CASAS_DECIMAIS = 6

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE_GPU = 64
BATCH_SIZE_CPU = 16
BATCH_SIZE = BATCH_SIZE_GPU if DEVICE.type == "cuda" else BATCH_SIZE_CPU

USAR_MIXED_PRECISION = True
BAIXAR_CSV_AUTOMATICAMENTE = True

if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    nome_gpu = torch.cuda.get_device_name(0)
    memoria_gpu = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✓ GPU detectada: {nome_gpu} ({memoria_gpu:.1f} GB)")
else:
    print("⚠ ATENÇÃO: nenhuma GPU foi detectada.")
    print("  O mDeBERTa-v3-base com NLI bidirecional em CPU pode levar horas.")
    print("  No Google Colab: Ambiente de execução > Alterar tipo de ambiente > T4 GPU.")
    torch.set_num_threads(min(8, os.cpu_count() or 1))

print(f"Dispositivo: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Modelo NLI: {MODEL_NAME}")

✓ GPU detectada: Tesla T4 (14.6 GB)
Dispositivo: cuda
Batch size: 64
Modelo NLI: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli


In [ ]:
# ============================================================
# 3. UPLOAD E IDENTIFICAÇÃO AUTOMÁTICA DOS JSONs
# ============================================================

def carregar_json_bytes(nome_arquivo, conteudo):
    try:
        texto = conteudo.decode("utf-8-sig")
        return json.loads(texto)
    except Exception as e:
        raise ValueError(f"Não foi possível ler '{nome_arquivo}' como JSON: {e}") from e


def classificar_json(nome_arquivo, dados):
    if not isinstance(dados, dict):
        return "desconhecido"

    casos = dados.get("cases")
    if not isinstance(casos, list):
        return "desconhecido"

    if dados.get("is_reference_base") is True:
        return "ground_truth"

    if casos:
        primeiro = casos[0]
        if isinstance(primeiro, dict):
            if "generations" in primeiro:
                return "geracoes"
            if "reference_id" in primeiro and "gherkin" in primeiro:
                return "ground_truth"

    return "desconhecido"


def processar_upload(uploaded):
    arquivos = []

    for nome, conteudo in uploaded.items():
        if not nome.lower().endswith(".json"):
            print(f"⚠ Ignorado (não é JSON): {nome}")
            continue

        dados = carregar_json_bytes(nome, conteudo)
        tipo = classificar_json(nome, dados)
        arquivos.append({"nome": nome, "tipo": tipo, "dados": dados})

    return arquivos


try:
    from google.colab import files
except ImportError as e:
    raise RuntimeError(
        "Este notebook foi preparado para upload interativo no Google Colab. "
        "Execute-o no Colab ou adapte esta célula para leitura local."
    ) from e


# ------------------------------------------------------------
# ETAPA 1 — Ground truth
# ------------------------------------------------------------
print("ETAPA 1/2 — Envie o arquivo JSON do ground truth:")
upload_gt = files.upload()
arquivos_json = processar_upload(upload_gt)

ground_truths = [a for a in arquivos_json if a["tipo"] == "ground_truth"]
arquivos_geracoes = [a for a in arquivos_json if a["tipo"] == "geracoes"]
desconhecidos = [a["nome"] for a in arquivos_json if a["tipo"] == "desconhecido"]

if desconhecidos:
    print("⚠ JSON(s) com estrutura não reconhecida:", desconhecidos)

if len(ground_truths) != 1:
    raise ValueError(
        f"É necessário exatamente 1 ground truth. Foram identificados {len(ground_truths)}. "
        "Verifique se o arquivo possui 'is_reference_base': true ou casos com "
        "'reference_id' e 'gherkin'."
    )

ground_truth_nome = ground_truths[0]["nome"]
ground_truth = ground_truths[0]["dados"]

print(f"\n✓ Ground truth identificado: {ground_truth_nome}")


# ------------------------------------------------------------
# ETAPA 2 — Gerações
# ------------------------------------------------------------
if not arquivos_geracoes:
    print("\nETAPA 2/2 — Agora envie um ou mais JSONs de gerações:")
    upload_gen = files.upload()
    novos_arquivos = processar_upload(upload_gen)

    novos_ground_truths = [a for a in novos_arquivos if a["tipo"] == "ground_truth"]
    if novos_ground_truths:
        print(
            "⚠ Ground truth adicional ignorado na etapa de gerações:",
            [a["nome"] for a in novos_ground_truths]
        )

    novos_desconhecidos = [
        a["nome"] for a in novos_arquivos if a["tipo"] == "desconhecido"
    ]
    if novos_desconhecidos:
        print("⚠ JSON(s) com estrutura não reconhecida:", novos_desconhecidos)

    arquivos_geracoes.extend(
        a for a in novos_arquivos if a["tipo"] == "geracoes"
    )

if not arquivos_geracoes:
    raise ValueError(
        "Nenhum arquivo de gerações foi identificado. Os arquivos de gerações "
        "devem possuir 'cases' e, dentro de cada caso, a chave 'generations'."
    )

print(f"\n✓ Arquivos de gerações identificados: {len(arquivos_geracoes)}")
for arq in arquivos_geracoes:
    dados = arq["dados"]
    print(
        f"  - {arq['nome']} | modelo={dados.get('model')} | "
        f"técnica={dados.get('technique')} | "
        f"execuções declaradas={dados.get('number_of_executions')}"
    )

ETAPA 1/2 — Envie o arquivo JSON do ground truth:


Saving base_referencia_gherkin.json to base_referencia_gherkin.json

✓ Ground truth identificado: base_referencia_gherkin.json

ETAPA 2/2 — Agora envie um ou mais JSONs de gerações:


Saving geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json to geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json

✓ Arquivos de gerações identificados: 1
  - geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json | modelo=ibm-granite/granite-4.1-8b | técnica=few-shot | execuções declaradas=10


In [ ]:
# ============================================================
# 4. VALIDAÇÃO DOS DADOS
# ============================================================

def indexar_ground_truth(dados_ground_truth):
    refs = {}
    duplicados = []

    for caso in dados_ground_truth.get("cases", []):
        case_id = caso.get("case_id")

        if not case_id:
            continue

        if case_id in refs:
            duplicados.append(case_id)

        refs[case_id] = caso

    if duplicados:
        raise ValueError(
            f"case_id duplicado(s) no ground truth: {sorted(set(duplicados))}"
        )

    return refs


referencias = indexar_ground_truth(ground_truth)
avisos_validacao = []

for arq in arquivos_geracoes:
    nome = arq["nome"]
    dados = arq["dados"]
    ids_arquivo = []
    declaradas = dados.get("number_of_executions")

    for caso in dados.get("cases", []):
        case_id = caso.get("case_id")
        ids_arquivo.append(case_id)

        if case_id not in referencias:
            avisos_validacao.append(
                f"{nome}: {case_id} existe nas gerações, mas não no ground truth."
            )
            continue

        original_ref = referencias[case_id].get("original_case")
        original_gen = caso.get("original_case")

        if (
            original_ref is not None
            and original_gen is not None
            and original_ref != original_gen
        ):
            avisos_validacao.append(
                f"{nome}: original_case divergente em {case_id}."
            )

        generations = caso.get("generations", [])

        if declaradas is not None and len(generations) != declaradas:
            avisos_validacao.append(
                f"{nome}: {case_id} possui {len(generations)} gerações, "
                f"mas o arquivo declara {declaradas}."
            )

        execucoes = [g.get("execution") for g in generations]
        execucoes_validas = [e for e in execucoes if e is not None]

        if len(execucoes_validas) != len(set(execucoes_validas)):
            avisos_validacao.append(
                f"{nome}: há números de execução duplicados em {case_id}."
            )

    ids_ref = set(referencias)
    ids_gen = set(ids_arquivo)

    ausentes = sorted(ids_ref - ids_gen)
    if ausentes:
        avisos_validacao.append(
            f"{nome}: {len(ausentes)} case_id(s) do ground truth não aparecem nas gerações. "
            f"Exemplos: {ausentes[:10]}"
        )

print(f"Casos no ground truth: {len(referencias)}")

if avisos_validacao:
    print(f"\n⚠ Foram encontrados {len(avisos_validacao)} aviso(s) de validação:")
    for aviso in avisos_validacao:
        print(" -", aviso)
else:
    print("\n✓ Estrutura validada sem avisos.")

Casos no ground truth: 259

✓ Estrutura validada sem avisos.


In [ ]:
# ============================================================
# 5. CARREGAMENTO DO MODELO NLI
# ============================================================

print("Carregando tokenizer e modelo NLI...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

def normalizar_nome_label(label):
    return str(label).strip().lower().replace("-", "_").replace(" ", "_")

def obter_indices_nli(config):
    id2label = getattr(config, "id2label", {}) or {}
    label2id = getattr(config, "label2id", {}) or {}
    mapa = {}

    for idx, nome in id2label.items():
        nome_norm = normalizar_nome_label(nome)
        if "entail" in nome_norm:
            mapa["entailment"] = int(idx)
        elif "neutral" in nome_norm:
            mapa["neutral"] = int(idx)
        elif "contrad" in nome_norm:
            mapa["contradiction"] = int(idx)

    for nome, idx in label2id.items():
        nome_norm = normalizar_nome_label(nome)
        if "entail" in nome_norm:
            mapa["entailment"] = int(idx)
        elif "neutral" in nome_norm:
            mapa["neutral"] = int(idx)
        elif "contrad" in nome_norm:
            mapa["contradiction"] = int(idx)

    faltantes = {"entailment", "neutral", "contradiction"} - set(mapa)
    if faltantes:
        raise ValueError(
            "Não foi possível identificar automaticamente todas as labels NLI. "
            f"Faltantes: {sorted(faltantes)} | "
            f"id2label={id2label} | label2id={label2id}"
        )

    return mapa

IDX_NLI = obter_indices_nli(model.config)

print("✓ Modelo carregado.")
print("Mapeamento NLI:", IDX_NLI)

Carregando tokenizer e modelo NLI...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

✓ Modelo carregado.
Mapeamento NLI: {'entailment': 0, 'neutral': 1, 'contradiction': 2}


In [ ]:
# ============================================================
# 6. FUNÇÕES: NORMALIZAÇÃO, GHERKIN, ALINHAMENTO E NLI
# ============================================================

PADRAO_PASSO = re.compile(
    r"^\s*(Given|When|Then|And|But|Dado|Dada|Dados|Dadas|Quando|Então|Entao|E|Mas)\b\s*(.*)$",
    flags=re.IGNORECASE
)

MAPA_GRUPO = {
    "given": "Given",
    "dado": "Given",
    "dada": "Given",
    "dados": "Given",
    "dadas": "Given",
    "when": "When",
    "quando": "When",
    "then": "Then",
    "então": "Then",
    "entao": "Then",
}

CONTINUACOES = {"and", "but", "e", "mas"}


def normalizar_gherkin(valor):
    """
    Converte diferentes representações de Gherkin em texto multilinha.

    Suporta:
    - string normal com quebras de linha;
    - string contendo \\n literal;
    - listas;
    - dicionários/objetos JSON;
    - blocos Markdown ```gherkin ... ```.
    """
    if valor is None:
        return ""

    if isinstance(valor, str):
        texto = valor

        # Converte sequências literais "\n" em quebras reais,
        # somente quando não há quebras reais relevantes.
        if "\\n" in texto:
            texto = texto.replace("\\r\\n", "\n").replace("\\n", "\n")

        # Remove cercas Markdown sem remover o conteúdo.
        texto = re.sub(
            r"^\s*```(?:gherkin|cucumber|feature)?\s*$",
            "",
            texto,
            flags=re.IGNORECASE | re.MULTILINE
        )
        texto = re.sub(
            r"^\s*```\s*$",
            "",
            texto,
            flags=re.MULTILINE
        )

        return texto.strip()

    if isinstance(valor, list):
        partes = [normalizar_gherkin(item) for item in valor]
        return "\n".join(p for p in partes if p).strip()

    if isinstance(valor, dict):
        # Prioriza chaves que normalmente carregam texto Gherkin.
        prioridades = [
            "gherkin",
            "text",
            "texto",
            "content",
            "conteudo",
            "scenario",
            "cenario",
            "steps",
            "passos",
            "given",
            "when",
            "then",
            "and",
        ]

        partes = []

        for chave in prioridades:
            if chave in valor:
                normalizado = normalizar_gherkin(valor[chave])
                if normalizado:
                    partes.append(normalizado)

        if partes:
            return "\n".join(partes).strip()

        # Fallback: percorre todos os valores.
        for item in valor.values():
            normalizado = normalizar_gherkin(item)
            if normalizado:
                partes.append(normalizado)

        return "\n".join(partes).strip()

    return str(valor).strip()


def extrair_passos_gherkin(valor):
    """
    Extrai passos Gherkin de diferentes formatos.

    And/But/E/Mas herdam o último contexto principal:
    Given, When ou Then.
    """
    texto = normalizar_gherkin(valor)

    passos = []
    grupo_atual = None
    posicoes_grupo = defaultdict(int)

    for linha in texto.splitlines():
        linha = linha.strip()

        # Remove marcadores comuns sem afetar a palavra-chave.
        linha = re.sub(r"^\s*[-*•]\s*", "", linha)

        m = PADRAO_PASSO.match(linha)
        if not m:
            continue

        keyword_original = m.group(1).strip()
        conteudo = m.group(2).strip()
        keyword_norm = keyword_original.lower()

        if keyword_norm in MAPA_GRUPO:
            grupo_atual = MAPA_GRUPO[keyword_norm]
        elif keyword_norm in CONTINUACOES:
            grupo_atual = grupo_atual or "SemContexto"
        else:
            grupo_atual = grupo_atual or "SemContexto"

        posicoes_grupo[grupo_atual] += 1

        passos.append({
            "ordem_global": len(passos) + 1,
            "keyword": keyword_original,
            "grupo": grupo_atual,
            "posicao_no_grupo": posicoes_grupo[grupo_atual],
            "texto": conteudo,
        })

    return passos


def alinhar_passos(passos_ref, passos_gen):
    ref_por_chave = {
        (p["grupo"], p["posicao_no_grupo"]): p
        for p in passos_ref
    }
    gen_por_chave = {
        (p["grupo"], p["posicao_no_grupo"]): p
        for p in passos_gen
    }

    ordem_grupos = {
        "Given": 0,
        "When": 1,
        "Then": 2,
        "SemContexto": 3
    }

    chaves = sorted(
        set(ref_por_chave) | set(gen_por_chave),
        key=lambda x: (ordem_grupos.get(x[0], 4), x[1])
    )

    pares = []
    faltantes = 0
    extras = 0

    for chave in chaves:
        ref = ref_por_chave.get(chave)
        gen = gen_por_chave.get(chave)

        if ref is not None and gen is not None:
            pares.append({
                "grupo": chave[0],
                "posicao_no_grupo": chave[1],
                "passo_ref": ref,
                "passo_gen": gen,
            })
        elif ref is not None:
            faltantes += 1
        elif gen is not None:
            extras += 1

    return pares, faltantes, extras


@torch.inference_mode()
def inferir_nli_pares(pares_texto, batch_size=None, descricao="NLI"):
    if not pares_texto:
        return []

    batch_size = batch_size or BATCH_SIZE
    saidas = []

    total_batches = (len(pares_texto) + batch_size - 1) // batch_size

    for inicio in tqdm(
        range(0, len(pares_texto), batch_size),
        total=total_batches,
        desc=descricao
    ):
        lote = pares_texto[inicio:inicio + batch_size]
        lote_p = [p for p, _ in lote]
        lote_h = [h for _, h in lote]

        inputs = tokenizer(
            lote_p,
            lote_h,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in inputs.items()
        }

        if DEVICE.type == "cuda" and USAR_MIXED_PRECISION:
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(**inputs).logits
        else:
            logits = model(**inputs).logits

        probs = torch.softmax(
            logits.float(),
            dim=-1
        ).cpu().numpy()

        for p in probs:
            pe = float(p[IDX_NLI["entailment"]])
            pn = float(p[IDX_NLI["neutral"]])
            pc = float(p[IDX_NLI["contradiction"]])

            score = pe + 0.5 * pn

            probabilidades = {
                "entailment": pe,
                "neutral": pn,
                "contradiction": pc,
            }

            saidas.append({
                "entailment": pe,
                "neutral": pn,
                "contradiction": pc,
                "nli_score": score,
                "classe_nli": max(
                    probabilidades,
                    key=probabilidades.get
                ),
            })

    return saidas


def combinar_nli_bidirecional(
    nli_ref_gen,
    nli_gen_ref
):
    pe = (
        nli_ref_gen["entailment"]
        + nli_gen_ref["entailment"]
    ) / 2

    pn = (
        nli_ref_gen["neutral"]
        + nli_gen_ref["neutral"]
    ) / 2

    pc = (
        nli_ref_gen["contradiction"]
        + nli_gen_ref["contradiction"]
    ) / 2

    score = (
        nli_ref_gen["nli_score"]
        + nli_gen_ref["nli_score"]
    ) / 2

    probabilidades = {
        "entailment": pe,
        "neutral": pn,
        "contradiction": pc,
    }

    return {
        "entailment": pe,
        "neutral": pn,
        "contradiction": pc,
        "nli_score": score,
        "classe_nli": max(
            probabilidades,
            key=probabilidades.get
        ),
        "nli_ref_para_gerado": nli_ref_gen["nli_score"],
        "nli_gerado_para_ref": nli_gen_ref["nli_score"],
    }


# Teste interno obrigatório da regex.
_teste = """Scenario: teste
Given uma condição
And outra condição
When uma ação
Then um resultado"""

_passos_teste = extrair_passos_gherkin(_teste)

assert len(_passos_teste) == 4, (
    "Falha interna no extrator Gherkin. "
    f"Foram encontrados {len(_passos_teste)} passos no teste."
)

print("✓ Extrator Gherkin validado internamente.")

✓ Extrator Gherkin validado internamente.


In [ ]:
# ============================================================
# 7. PREPARAÇÃO DAS COMPARAÇÕES POR PASSOS
# ============================================================

geracoes_base = []
pares_passos_base = []

# Mostra amostra real antes de processar tudo.
primeira_ref = next(iter(referencias.values()))

print("AMOSTRA DA ESTRUTURA DO GROUND TRUTH")
print("Tipo do campo gherkin:", type(primeira_ref.get("gherkin")).__name__)
print(
    normalizar_gherkin(primeira_ref.get("gherkin"))[:1000]
)
print("-" * 80)

primeira_gen_mostrada = False

for arq in arquivos_geracoes:
    nome_geracoes = arq["nome"]
    dados = arq["dados"]
    modelo_nome = dados.get("model", "")
    tecnica = dados.get("technique", "")
    execucoes_declaradas = dados.get(
        "number_of_executions"
    )

    for caso_gerado in dados.get("cases", []):
        case_id = caso_gerado.get("case_id")
        referencia = referencias.get(case_id)

        if referencia is None:
            continue

        valor_ref = referencia.get("gherkin")
        gherkin_ref = normalizar_gherkin(valor_ref)
        passos_ref = extrair_passos_gherkin(valor_ref)

        for geracao in caso_gerado.get(
            "generations",
            []
        ):
            valor_gen = geracao.get("gherkin")
            gherkin_gen = normalizar_gherkin(valor_gen)
            passos_gen = extrair_passos_gherkin(valor_gen)

            if not primeira_gen_mostrada:
                print(
                    "AMOSTRA DA ESTRUTURA DE UMA GERAÇÃO"
                )
                print(
                    "Tipo do campo gherkin:",
                    type(valor_gen).__name__
                )
                print(gherkin_gen[:1000])
                print("-" * 80)
                primeira_gen_mostrada = True

            pares, passos_ausentes, passos_extras = (
                alinhar_passos(
                    passos_ref,
                    passos_gen
                )
            )

            id_interno = len(geracoes_base)

            coverage = (
                len(pares) / len(passos_ref)
                if len(passos_ref) > 0
                else np.nan
            )

            geracoes_base.append({
                "_id_interno": id_interno,
                "arquivo_ground_truth": ground_truth_nome,
                "arquivo_geracoes": nome_geracoes,
                "modelo": modelo_nome,
                "tecnica": tecnica,
                "execucoes_declaradas": execucoes_declaradas,
                "case_id": case_id,
                "source_id": referencia.get("source_id"),
                "source_line": referencia.get("source_line"),
                "original_case": referencia.get(
                    "original_case"
                ),
                "reference_id": referencia.get(
                    "reference_id"
                ),
                "generation_id": geracao.get(
                    "generation_id"
                ),
                "execucao": geracao.get("execution"),
                "gherkin_ground_truth": gherkin_ref,
                "gherkin_gerado": gherkin_gen,
                "passos_ground_truth": len(passos_ref),
                "passos_gerados": len(passos_gen),
                "passos_alinhados": len(pares),
                "passos_ausentes": passos_ausentes,
                "passos_extras": passos_extras,
                "coverage": coverage,
            })

            for par in pares:
                pares_passos_base.append({
                    "_id_interno": id_interno,
                    "arquivo_geracoes": nome_geracoes,
                    "modelo": modelo_nome,
                    "tecnica": tecnica,
                    "case_id": case_id,
                    "generation_id": geracao.get(
                        "generation_id"
                    ),
                    "execucao": geracao.get(
                        "execution"
                    ),
                    "grupo": par["grupo"],
                    "posicao_no_grupo": par[
                        "posicao_no_grupo"
                    ],
                    "keyword_ground_truth": par[
                        "passo_ref"
                    ]["keyword"],
                    "keyword_gerado": par[
                        "passo_gen"
                    ]["keyword"],
                    "passo_ground_truth": par[
                        "passo_ref"
                    ]["texto"],
                    "passo_gerado": par[
                        "passo_gen"
                    ]["texto"],
                })

if not geracoes_base:
    raise ValueError(
        "Nenhuma geração pôde ser preparada."
    )

print(
    f"✓ Gerações preparadas: "
    f"{len(geracoes_base):,}"
)
print(
    f"✓ Pares de passos alinhados: "
    f"{len(pares_passos_base):,}"
)

total_passos_gt = sum(
    x["passos_ground_truth"]
    for x in geracoes_base
)
total_passos_gen = sum(
    x["passos_gerados"]
    for x in geracoes_base
)
total_alinhados = sum(
    x["passos_alinhados"]
    for x in geracoes_base
)

print()
print("VALIDAÇÃO DA EXTRAÇÃO GHERKIN")
print(
    f"Total de passos no ground truth: "
    f"{total_passos_gt:,}"
)
print(
    f"Total de passos nas gerações: "
    f"{total_passos_gen:,}"
)
print(
    f"Total de passos alinhados: "
    f"{total_alinhados:,}"
)

if total_passos_gt == 0:
    raise RuntimeError(
        "O campo gherkin do ground truth existe, "
        "mas nenhum Given/When/Then/And foi reconhecido. "
        "Veja a AMOSTRA impressa acima."
    )

if total_passos_gen == 0:
    raise RuntimeError(
        "O campo gherkin das gerações existe, "
        "mas nenhum Given/When/Then/And foi reconhecido. "
        "Veja a AMOSTRA impressa acima."
    )

if total_alinhados == 0:
    raise RuntimeError(
        "Os passos foram reconhecidos, mas nenhum "
        "pôde ser alinhado entre referência e geração."
    )

print(
    "✓ Extração e alinhamento validados. "
    "Pode executar a célula 8."
)

AMOSTRA DA ESTRUTURA DO GROUND TRUTH
Tipo do campo gherkin: str
Scenario: Cancel editing project variable
  Given que o usuário está editando uma variável do projeto
  When ele cancela a edição
  Then a edição da variável do projeto é cancelada
--------------------------------------------------------------------------------
AMOSTRA DA ESTRUTURA DE UMA GERAÇÃO
Tipo do campo gherkin: str
Scenario: Cancelar edição da variável do projeto
  Given o usuário está editando uma variável do projeto
  When o usuário clica no botão "Cancelar"
  Then a edição da variável é revertida ao estado anterior
    And o formulário de edição é fechado
--------------------------------------------------------------------------------
✓ Gerações preparadas: 2,590
✓ Pares de passos alinhados: 7,770

VALIDAÇÃO DA EXTRAÇÃO GHERKIN
Total de passos no ground truth: 7,770
Total de passos nas gerações: 12,857
Total de passos alinhados: 7,770
✓ Extração e alinhamento validados. Pode executar a célula 8.


In [ ]:
# ============================================================
# 8. NLI BIDIRECIONAL OTIMIZADO POR PASSOS
# ============================================================

if DEVICE.type == "cpu":
    print("⚠ Você está executando em CPU.")
    print("  Para este conjunto de dados, recomenda-se fortemente usar GPU.")
    print()

pares_direcionais = []

for item in pares_passos_base:
    ref = item["passo_ground_truth"]
    gen = item["passo_gerado"]

    pares_direcionais.append((ref, gen))
    pares_direcionais.append((gen, ref))

total_sem_cache = len(pares_direcionais)

# Remove somente pares EXATAMENTE repetidos.
# O resultado matemático não é alterado.
pares_unicos = list(dict.fromkeys(pares_direcionais))

economia = total_sem_cache - len(pares_unicos)
percentual_economia = (
    100 * economia / total_sem_cache
    if total_sem_cache > 0
    else 0
)

print(f"Pares alinhados: {len(pares_passos_base):,}")
print(f"Inferências bidirecionais sem cache: {total_sem_cache:,}")
print(f"Pares direcionais únicos: {len(pares_unicos):,}")
print(
    f"Cache eliminou {economia:,} inferências repetidas "
    f"({percentual_economia:.1f}%)."
)
print(f"Dispositivo utilizado: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print()

inicio_calculo = time.time()

resultados_unicos = inferir_nli_pares(
    pares_unicos,
    batch_size=BATCH_SIZE,
    descricao="NLI bidirecional"
)

tempo_total = time.time() - inicio_calculo

cache_nli = {
    par: resultado
    for par, resultado in zip(pares_unicos, resultados_unicos)
}

print()
print(f"✓ Inferência concluída em {tempo_total / 60:.2f} minutos.")

detalhes_passos = []
por_geracao = defaultdict(list)

for base in pares_passos_base:
    ref = base["passo_ground_truth"]
    gen = base["passo_gerado"]

    rg = cache_nli[(ref, gen)]
    gr = cache_nli[(gen, ref)]

    nli_bi = combinar_nli_bidirecional(rg, gr)

    registro = {
        **base,
        **nli_bi,
    }

    detalhes_passos.append(registro)
    por_geracao[base["_id_interno"]].append(nli_bi)

resultados = []

for base in geracoes_base:
    scores_passos = por_geracao.get(base["_id_interno"], [])

    if scores_passos:
        nli_score = float(np.mean([
            x["nli_score"]
            for x in scores_passos
        ]))

        entailment_mean = float(np.mean([
            x["entailment"]
            for x in scores_passos
        ]))

        neutral_mean = float(np.mean([
            x["neutral"]
            for x in scores_passos
        ]))

        contradiction_mean = float(np.mean([
            x["contradiction"]
            for x in scores_passos
        ]))

        contradiction_rate = float(np.mean([
            x["classe_nli"] == "contradiction"
            for x in scores_passos
        ]))
    else:
        nli_score = np.nan
        entailment_mean = np.nan
        neutral_mean = np.nan
        contradiction_mean = np.nan
        contradiction_rate = np.nan

    def arred(v):
        return (
            round(v, NLI_CASAS_DECIMAIS)
            if not np.isnan(v)
            else np.nan
        )

    resultados.append({
        **{
            k: v
            for k, v in base.items()
            if k != "_id_interno"
        },
        "nli_score": arred(nli_score),
        "entailment_mean": arred(entailment_mean),
        "neutral_mean": arred(neutral_mean),
        "contradiction_mean": arred(contradiction_mean),
        "contradiction_rate": arred(contradiction_rate),
    })

df_resultados = pd.DataFrame(resultados)
df_detalhes_passos = pd.DataFrame(detalhes_passos)

if not df_detalhes_passos.empty:
    for col in [
        "entailment",
        "neutral",
        "contradiction",
        "nli_score",
        "nli_ref_para_gerado",
        "nli_gerado_para_ref",
    ]:
        df_detalhes_passos[col] = (
            df_detalhes_passos[col]
            .round(NLI_CASAS_DECIMAIS)
        )

chaves_ranking = [
    "arquivo_geracoes",
    "modelo",
    "tecnica",
    "case_id",
]

df_resultados = df_resultados.sort_values(
    chaves_ranking + ["nli_score", "execucao"],
    ascending=[True, True, True, True, False, True],
    kind="stable",
    na_position="last"
).reset_index(drop=True)

df_resultados["ranking_no_caso"] = (
    df_resultados
    .groupby(chaves_ranking, dropna=False)
    .cumcount()
    + 1
)

colunas = [
    "modelo",
    "tecnica",
    "case_id",
    "source_id",
    "original_case",
    "execucao",
    "nli_score",
    "ranking_no_caso",
    "entailment_mean",
    "neutral_mean",
    "contradiction_mean",
    "contradiction_rate",
    "coverage",
    "passos_ground_truth",
    "passos_gerados",
    "passos_alinhados",
    "passos_ausentes",
    "passos_extras",
    "generation_id",
    "reference_id",
    "gherkin_ground_truth",
    "gherkin_gerado",
    "execucoes_declaradas",
    "arquivo_ground_truth",
    "arquivo_geracoes",
]

df_resultados = df_resultados[colunas]

print()
print(f"✓ Comparações calculadas: {len(df_resultados):,}")
print(f"✓ Casos avaliados: {df_resultados['case_id'].nunique():,}")
print(
    "✓ Cenários sem passos alinhados: "
    f"{df_resultados['nli_score'].isna().sum():,}"
)

Pares alinhados: 7,770
Inferências bidirecionais sem cache: 15,540
Pares direcionais únicos: 5,667
Cache eliminou 9,873 inferências repetidas (63.5%).
Dispositivo utilizado: cuda
Batch size: 64



NLI bidirecional:   0%|          | 0/89 [00:00<?, ?it/s]


✓ Inferência concluída em 0.15 minutos.

✓ Comparações calculadas: 2,590
✓ Casos avaliados: 259
✓ Cenários sem passos alinhados: 0


In [ ]:
# ============================================================
# 9. TABELAS DE ANÁLISE
# ============================================================

df_resumo_geral = (
    df_resultados
    .groupby(["arquivo_geracoes", "modelo", "tecnica"], dropna=False)
    .agg(
        casos=("case_id", "nunique"),
        comparacoes=("nli_score", "count"),
        nli_media=("nli_score", "mean"),
        nli_mediana=("nli_score", "median"),
        desvio_padrao=("nli_score", "std"),
        nli_minimo=("nli_score", "min"),
        nli_maximo=("nli_score", "max"),
        entailment_medio=("entailment_mean", "mean"),
        neutral_medio=("neutral_mean", "mean"),
        contradiction_medio=("contradiction_mean", "mean"),
        taxa_contradicao_media=("contradiction_rate", "mean"),
        coverage_medio=("coverage", "mean"),
    )
    .reset_index()
)

print("RESUMO GERAL")
display(df_resumo_geral.style.format({
    "nli_media": "{:.4f}", "nli_mediana": "{:.4f}", "desvio_padrao": "{:.4f}",
    "nli_minimo": "{:.4f}", "nli_maximo": "{:.4f}", "entailment_medio": "{:.4f}",
    "neutral_medio": "{:.4f}", "contradiction_medio": "{:.4f}",
    "taxa_contradicao_media": "{:.2%}", "coverage_medio": "{:.2%}",
}))


df_resumo_casos = (
    df_resultados
    .groupby(["arquivo_geracoes", "modelo", "tecnica", "case_id", "original_case"], dropna=False)
    .agg(
        execucoes_avaliadas=("execucao", "count"),
        nli_media=("nli_score", "mean"),
        nli_mediana=("nli_score", "median"),
        desvio_padrao=("nli_score", "std"),
        melhor_nli=("nli_score", "max"),
        pior_nli=("nli_score", "min"),
        contradiction_media=("contradiction_mean", "mean"),
        contradiction_rate_media=("contradiction_rate", "mean"),
        coverage_media=("coverage", "mean"),
    )
    .reset_index()
    .sort_values(["modelo", "tecnica", "case_id"])
)

df_resumo_casos["cv_nli_percentual"] = np.where(
    df_resumo_casos["nli_media"] != 0,
    (df_resumo_casos["desvio_padrao"] / df_resumo_casos["nli_media"]) * 100,
    np.nan
)

print("\nRESUMO POR CASO — primeiras 30 linhas")
display(df_resumo_casos.head(30).style.format({
    "nli_media": "{:.4f}", "nli_mediana": "{:.4f}", "desvio_padrao": "{:.4f}",
    "melhor_nli": "{:.4f}", "pior_nli": "{:.4f}", "contradiction_media": "{:.4f}",
    "contradiction_rate_media": "{:.2%}", "coverage_media": "{:.2%}",
    "cv_nli_percentual": "{:.2f}%",
}))

print("\nCOMPARAÇÕES — primeiras 50 linhas")
colunas_visualizacao = [
    "modelo", "tecnica", "case_id", "original_case", "execucao", "nli_score",
    "ranking_no_caso", "entailment_mean", "neutral_mean", "contradiction_mean",
    "contradiction_rate", "coverage", "passos_ground_truth", "passos_gerados",
    "passos_alinhados", "passos_ausentes", "passos_extras",
]
display(df_resultados[colunas_visualizacao].head(50).style.format({
    "nli_score": "{:.4f}", "entailment_mean": "{:.4f}", "neutral_mean": "{:.4f}",
    "contradiction_mean": "{:.4f}", "contradiction_rate": "{:.2%}", "coverage": "{:.2%}",
}))

RESUMO GERAL


,arquivo_geracoes,modelo,tecnica,casos,comparacoes,nli_media,nli_mediana,desvio_padrao,nli_minimo,nli_maximo,entailment_medio,neutral_medio,contradiction_medio,taxa_contradicao_media,coverage_medio
0,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,259,2590,0.6458,0.6604,0.1397,0.2229,0.9989,0.4069,0.4778,0.1153,10.59%,100.00%



RESUMO POR CASO — primeiras 30 linhas


,arquivo_geracoes,modelo,tecnica,case_id,original_case,execucoes_avaliadas,nli_media,nli_mediana,desvio_padrao,melhor_nli,pior_nli,contradiction_media,contradiction_rate_media,coverage_media,cv_nli_percentual
0,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,10,0.6408,0.6441,0.0207,0.6658,0.6070,0.1541,0.00%,100.00%,3.23%
1,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1002,Cadastrar transação com campos inválidos,10,0.5773,0.6188,0.0625,0.6384,0.5035,0.1834,13.33%,100.00%,10.83%
2,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1006,Verificar se todos os dados da transação estão sendo exibidos,10,0.7566,0.7616,0.0263,0.7797,0.7067,0.0005,0.00%,100.00%,3.47%
3,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1007,Verificar inserção de link da transação inexistente,10,0.3506,0.3323,0.0541,0.4234,0.3001,0.3506,33.33%,100.00%,15.44%
4,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_101,Validar resultado de consulta de Matéria-Prima vazia,10,0.5694,0.5548,0.1014,0.7355,0.3927,0.1645,26.67%,100.00%,17.80%
5,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1030,Cadastar categoria com sucesso,10,0.7709,0.7817,0.0321,0.7827,0.6798,0.0998,0.00%,100.00%,4.17%
6,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1033,Cadastrar categoria com campos inválidos,10,0.6644,0.6528,0.1075,0.8364,0.4989,0.1663,13.33%,100.00%,16.18%
7,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1035,Editar categoria deixando os campos obrigatórios do formulário em branco,10,0.5306,0.5164,0.0471,0.6371,0.4522,0.1541,6.67%,100.00%,8.87%
8,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1060,Cancelar cadastro de conta com sucesso,10,0.4285,0.4292,0.0074,0.4408,0.4101,0.3343,33.33%,100.00%,1.73%
9,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1061,Cadastrar conta com campos obrigatórios não preenchidos,10,0.6548,0.6289,0.0418,0.7174,0.6289,0.1312,23.33%,100.00%,6.38%



COMPARAÇÕES — primeiras 50 linhas


,modelo,tecnica,case_id,original_case,execucao,nli_score,ranking_no_caso,entailment_mean,neutral_mean,contradiction_mean,contradiction_rate,coverage,passos_ground_truth,passos_gerados,passos_alinhados,passos_ausentes,passos_extras
0,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,4,0.6658,1,0.4905,0.3506,0.1590,0.00%,100.00%,3,5,3,0,2
1,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,5,0.6656,2,0.4902,0.3509,0.1590,0.00%,100.00%,3,6,3,0,3
2,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,1,0.6615,3,0.4820,0.3590,0.1590,0.00%,100.00%,3,5,3,0,2
3,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,3,0.6486,4,0.4561,0.3850,0.1589,0.00%,100.00%,3,5,3,0,2
4,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,6,0.6441,5,0.4474,0.3935,0.1591,0.00%,100.00%,3,5,3,0,2
5,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,9,0.6441,6,0.4474,0.3935,0.1591,0.00%,100.00%,3,5,3,0,2
6,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,7,0.6271,7,0.3970,0.4604,0.1427,0.00%,100.00%,3,5,3,0,2
7,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,8,0.6271,8,0.3970,0.4604,0.1427,0.00%,100.00%,3,5,3,0,2
8,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,10,0.6167,9,0.3759,0.4815,0.1426,0.00%,100.00%,3,5,3,0,2
9,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,2,0.6070,10,0.3730,0.4680,0.1591,0.00%,100.00%,3,5,3,0,2


In [ ]:
# ============================================================
# 10. CONSULTA RÁPIDA DE UM CASO
# ============================================================

def visualizar_caso(case_id, mostrar_passos=True):
    recorte = df_resultados[df_resultados["case_id"] == case_id].copy()
    if recorte.empty:
        print(f"Nenhum resultado encontrado para {case_id}.")
        return

    colunas = [
        "modelo", "tecnica", "case_id", "original_case", "execucao", "nli_score",
        "ranking_no_caso", "entailment_mean", "neutral_mean", "contradiction_mean",
        "contradiction_rate", "coverage", "passos_ground_truth", "passos_gerados",
        "passos_alinhados", "passos_ausentes", "passos_extras",
        "gherkin_ground_truth", "gherkin_gerado",
    ]

    display(recorte[colunas].sort_values(
        ["modelo", "tecnica", "nli_score", "execucao"],
        ascending=[True, True, False, True]
    ).style.format({
        "nli_score": "{:.4f}", "entailment_mean": "{:.4f}", "neutral_mean": "{:.4f}",
        "contradiction_mean": "{:.4f}", "contradiction_rate": "{:.2%}", "coverage": "{:.2%}",
    }))

    if mostrar_passos and not df_detalhes_passos.empty:
        detalhes = df_detalhes_passos[df_detalhes_passos["case_id"] == case_id].copy()
        if not detalhes.empty:
            print("\nNLI BIDIRECIONAL POR PASSO")
            colunas_passos = [
                "modelo", "tecnica", "case_id", "execucao", "grupo", "posicao_no_grupo",
                "passo_ground_truth", "passo_gerado", "nli_ref_para_gerado",
                "nli_gerado_para_ref", "nli_score", "entailment", "neutral",
                "contradiction", "classe_nli",
            ]
            display(detalhes[colunas_passos].sort_values(
                ["modelo", "tecnica", "execucao", "grupo", "posicao_no_grupo"]
            ).style.format({
                "nli_ref_para_gerado": "{:.4f}", "nli_gerado_para_ref": "{:.4f}",
                "nli_score": "{:.4f}", "entailment": "{:.4f}", "neutral": "{:.4f}",
                "contradiction": "{:.4f}",
            }))

primeiro_case_id = df_resultados["case_id"].iloc[0]
print(f"Exemplo de consulta: {primeiro_case_id}")
visualizar_caso(primeiro_case_id)

# visualizar_caso("TC_261")

Exemplo de consulta: TC_1


,modelo,tecnica,case_id,original_case,execucao,nli_score,ranking_no_caso,entailment_mean,neutral_mean,contradiction_mean,contradiction_rate,coverage,passos_ground_truth,passos_gerados,passos_alinhados,passos_ausentes,passos_extras,gherkin_ground_truth,gherkin_gerado
0,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,4,0.6658,1,0.4905,0.3506,0.1590,0.00%,100.00%,3,5,3,0,2,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidade And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação ""Entidade cadastrada com sucesso"" And o novo registro deve aparecer na lista de entidades existentes"
1,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,5,0.6656,2,0.4902,0.3509,0.1590,0.00%,100.00%,3,6,3,0,3,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidades And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação ""Entidade cadastrada com sucesso"" And o sistema deve armazenar a entidade no banco de dados And o usuário é redirecionado para a página de listagem de entidades"
2,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,1,0.6615,3,0.4820,0.3590,0.1590,0.00%,100.00%,3,5,3,0,2,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidades And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve confirmar a criação da entidade com uma mensagem de sucesso And o novo registro deve aparecer na lista de entidades cadastradas"
3,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,3,0.6486,4,0.4561,0.3850,0.1589,0.00%,100.00%,3,5,3,0,2,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro da entidade And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação de cadastro bem-sucedido And o novo registro deve aparecer na lista de entidades cadastradas"
4,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,6,0.6441,5,0.4474,0.3935,0.1591,0.00%,100.00%,3,5,3,0,2,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidades And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação indicando que a entidade foi cadastrada com sucesso And o novo registro deve aparecer na lista de entidades existentes"
5,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,9,0.6441,6,0.4474,0.3935,0.1591,0.00%,100.00%,3,5,3,0,2,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na 


NLI BIDIRECIONAL POR PASSO


,modelo,tecnica,case_id,execucao,grupo,posicao_no_grupo,passo_ground_truth,passo_gerado,nli_ref_para_gerado,nli_gerado_para_ref,nli_score,entailment,neutral,contradiction,classe_nli
3870,ibm-granite/granite-4.1-8b,few-shot,TC_1,1,Given,1,que o usuário está na funcionalidade de cadastro de entidade,o usuário está na página de cadastro de entidades,0.9942,0.9958,0.9950,0.9907,0.0086,0.0007,entailment
3872,ibm-granite/granite-4.1-8b,few-shot,TC_1,1,Then,1,a entidade é cadastrada com sucesso,o sistema deve confirmar a criação da entidade com uma mensagem de sucesso,0.5029,0.9393,0.7211,0.4425,0.5572,0.0003,neutral
3871,ibm-granite/granite-4.1-8b,few-shot,TC_1,1,When,1,ele preenche os dados e confirma o cadastro,"o usuário clica no botão ""Salvar""",0.0286,0.5083,0.2684,0.0129,0.5111,0.4760,neutral
3873,ibm-granite/granite-4.1-8b,few-shot,TC_1,2,Given,1,que o usuário está na funcionalidade de cadastro de entidade,o usuário está na página de cadastro de entidade,0.9953,0.9958,0.9956,0.9917,0.0077,0.0006,entailment
3875,ibm-granite/granite-4.1-8b,few-shot,TC_1,2,Then,1,a entidade é cadastrada com sucesso,o sistema deve validar os dados e exibir uma mensagem de sucesso,0.5012,0.6126,0.5569,0.1144,0.8851,0.0006,neutral
3874,ibm-granite/granite-4.1-8b,few-shot,TC_1,2,When,1,ele preenche os dados e confirma o cadastro,"o usuário clica no botão ""Salvar""",0.0286,0.5083,0.2684,0.0129,0.5111,0.4760,neutral
3876,ibm-granite/granite-4.1-8b,few-shot,TC_1,3,Given,1,que o usuário está na funcionalidade de cadastro de entidade,o usuário está na página de cadastro da entidade,0.9962,0.9966,0.9964,0.9932,0.0064,0.0004,entailment
3878,ibm-granite/granite-4.1-8b,few-shot,TC_1,3,Then,1,a entidade é cadastrada com sucesso,o sistema deve exibir uma mensagem de confirmação de cadastro bem-sucedido,0.5012,0.8609,0.6810,0.3623,0.6375,0.0002,neutral
3877,ibm-granite/granite-4.1-8b,few-shot,TC_1,3,When,1,ele preenche os dados e confirma o cadastro,"o usuário clica no botão ""Salvar""",0.0286,0.5083,0.2684,0.0129,0.5111,0.4760,neutral
3879,ibm-granite/granite-4.1-8b,few-shot,TC_1,4,Given,1,que o usuário está na funcionalidade de cadastro de entidade,o usuário está na página de cadastro de entidade,0.9953,0.9958,0.9956,0.9917,0.0077,0.0006,entailment


In [ ]:
# ============================================================
# 11. EXPORTAÇÃO — UM ÚNICO CSV
# ============================================================

def slug(texto):
    texto = str(texto or "").strip().lower()
    texto = re.sub(r"[^a-z0-9._-]+", "-", texto)
    texto = re.sub(r"-+", "-", texto).strip("-")
    return texto or "sem-identificacao"

metadados_unicos = df_resultados[["modelo", "tecnica"]].drop_duplicates().reset_index(drop=True)

if len(metadados_unicos) == 1:
    modelo_nome = metadados_unicos.loc[0, "modelo"]
    tecnica_nome = metadados_unicos.loc[0, "tecnica"]
    nome_csv = f"metricas_nli_passos_bidirecional_{slug(modelo_nome)}_{slug(tecnica_nome)}.csv"
else:
    nome_csv = "metricas_nli_passos_bidirecional_multiplos_modelos_tecnicas.csv"

df_resultados.to_csv(nome_csv, index=False, encoding="utf-8-sig")

print(f"✓ CSV gerado: {nome_csv}")
print(f"✓ Linhas exportadas: {len(df_resultados):,}")
print("✓ Somente o CSV agregado por cenário/execução é exportado.")

if BAIXAR_CSV_AUTOMATICAMENTE:
    try:
        from google.colab import files
        files.download(nome_csv)
    except Exception as e:
        print(f"Download automático não realizado: {e}")

✓ CSV gerado: metricas_nli_passos_bidirecional_ibm-granite-granite-4.1-8b_few-shot.csv
✓ Linhas exportadas: 2,590
✓ Somente o CSV agregado por cenário/execução é exportado.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>